# Chapter 9: Dictionary Mastery for VLSI Professionals

Companion notebook to **python-from-zero** · `lesson-09` · based on *Python for VLSI*, Chapter 9.

### You will learn

- Build and query dictionaries for O(1) lookups.
- Iterate over keys, values, and items cleanly.
- Use `.get`, `.setdefault`, and `defaultdict` idiomatically.
- Model fanouts, delay tables, and cell libraries as dicts.

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## Creating dicts

Key-value pairs keyed by hashable types. Use `{}` or `dict(...)`.

In [2]:
fanouts = {"clk": 4200, "reset": 800, "data": 150}
print(fanouts["clk"], len(fanouts))
print(dict(a=1, b=2))


4200 3
{'a': 1, 'b': 2}


## Safe lookup with .get

`.get(key, default)` returns `default` when the key is missing, never raising `KeyError`.

In [3]:
print(fanouts.get("clk"))
print(fanouts.get("missing", 0))


4200
0


## Iterating over dicts

Three common loops: keys, values, items.

In [4]:
for k in fanouts:                          # keys
    print("key  :", k)
for v in fanouts.values():                 # values
    print("val  :", v)
for k, v in fanouts.items():               # both
    print(f"{k:6s} -> {v}")


key  : clk
key  : reset
key  : data
val  : 4200
val  : 800
val  : 150
clk    -> 4200
reset  -> 800
data   -> 150


## Updating and merging

`d[k] = v` inserts/replaces. `d.update(other)` merges. Python 3.9+ allows `d1 | d2` for a merged copy.

In [5]:
library = {"INV_X1": 0.12, "NAND2_X1": 0.25}
library["DFF_X1"] = 1.10
library.update({"MUX2_X1": 0.40})
print(library)
merged = library | {"INV_X1": 0.10}
print("merged (INV_X1 overridden):", merged)


{'INV_X1': 0.12, 'NAND2_X1': 0.25, 'DFF_X1': 1.1, 'MUX2_X1': 0.4}
merged (INV_X1 overridden): {'INV_X1': 0.1, 'NAND2_X1': 0.25, 'DFF_X1': 1.1, 'MUX2_X1': 0.4}


## setdefault and defaultdict

For grouping, `setdefault` is a one-liner; `defaultdict` avoids the explicit `if key not in d:` dance entirely.

In [6]:
from collections import defaultdict

pairs = [("M1","NAND"), ("M1","INV"), ("M2","DFF"), ("M1","DFF")]
by_layer = defaultdict(list)
for layer, cell in pairs:
    by_layer[layer].append(cell)
for l, cs in by_layer.items():
    print(l, cs)


M1 ['NAND', 'INV', 'DFF']
M2 ['DFF']


## Dict comprehensions

Compact way to build a lookup.

In [7]:
cells = ["INV_X1", "NAND2_X1", "DFF_X1"]
area  = {c: i * 0.5 + 0.25 for i, c in enumerate(cells)}
print(area)


{'INV_X1': 0.25, 'NAND2_X1': 0.75, 'DFF_X1': 1.25}


## Counting with Counter

`collections.Counter` is a dict preconfigured for counting.

In [8]:
from collections import Counter
tokens = ["clk","data","clk","reset","clk","data"]
print(Counter(tokens).most_common())


[('clk', 3), ('data', 2), ('reset', 1)]


## VLSI practice — delay lookup table

Build a 2-level dict `{cell: {load: delay_ns}}` and query it safely.

In [9]:
delays = {
    "INV_X1":   {0.5: 0.045, 1.0: 0.070, 2.0: 0.120},
    "NAND2_X1": {0.5: 0.060, 1.0: 0.090, 2.0: 0.155},
}

def lookup(cell, load, default=None):
    return delays.get(cell, {}).get(load, default)

print("INV_X1  @ 1.0pF :", lookup("INV_X1", 1.0))
print("DFF_X1  @ 1.0pF :", lookup("DFF_X1", 1.0, default="N/A"))

# Flatten into a report
for cell, row in delays.items():
    for load, d in row.items():
        print(f"{cell:10s} load={load:.1f}pF delay={d:.3f}ns")


INV_X1  @ 1.0pF : 0.07
DFF_X1  @ 1.0pF : N/A
INV_X1     load=0.5pF delay=0.045ns
INV_X1     load=1.0pF delay=0.070ns
INV_X1     load=2.0pF delay=0.120ns
NAND2_X1   load=0.5pF delay=0.060ns
NAND2_X1   load=1.0pF delay=0.090ns
NAND2_X1   load=2.0pF delay=0.155ns


### Exercise 1
Build a dict from `['clk','data','rst']` mapping each to its length.

In [10]:
# Your code here


<details><summary>Show solution</summary>

```python
a=['clk','data','rst']; print({s:len(s) for s in a})
```

</details>

### Exercise 2
Given `d={'a':1,'b':2}`, return 0 for missing key `'c'` using `.get`.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
d={'a':1,'b':2}; print(d.get('c',0))
```

</details>

### Exercise 3
Count occurrences of each word in `'clk data clk rst data clk'` with `Counter`.

In [12]:
# Your code here


<details><summary>Show solution</summary>

```python
from collections import Counter; print(Counter('clk data clk rst data clk'.split()))
```

</details>

## Recap

- Dicts give O(1) lookup keyed by hashable values.
- Prefer `.get` over `[]` when missing keys are expected.
- `defaultdict` and `Counter` reduce boilerplate for grouping/counting.
- Comprehensions and the `|` operator build/merge dicts concisely.

## What's next

Continue with **Chapter 10: Tuple Mastery for VLSI Professionals** in this repo, and the matching lesson on [python-from-zero](https://python-from-zero.pages.dev).